# Experiment 9c - Chronos-2 (foundation model), zero-shot + stacking

Frontier 2 of [`../IMPROVEMENT-FRONTIERS.md`](../IMPROVEMENT-FRONTIERS.md). After
9a/9b closed the spike tail for WAPE (a metric-level dead end), the remaining lever
is **genuinely decorrelated signal**. Within your own model family residuals
correlate at r ~ 0.99 (LGBM/XGB/MLP all learned the same function from the same 52
features). A pretrained foundation model is the one source whose residuals should be
*structurally* different - different training data (billions of series), different
architecture, different everything.

**Why Chronos-2 specifically:**
- Native support for **known future covariates** (the 11 domain forecasts over the
  336h horizon) - the property worth ~10 WAPE on this dataset (APPROACH §6.7.1).
- `chronos-2-native-optimized` already **leads the Overall leaderboard** here, so it
  is empirically strong on *both* WAPE (14.81) and the L2 metrics we trail on.
- Encoder-only, 120M params, quantile outputs, zero-shot or fine-tunable.

**Two questions this notebook answers, in order:**
1. **Decorrelation** - is `r(Chronos resid, LGBM resid)` and `r(Chronos, MLP)` below
   0.85? If yes, Chronos is a real stacking arm regardless of its solo WAPE.
2. **Blend WAPE** - does an OOF-tuned blend of Chronos + your gated-MLP beat the
   14.47 ensemble? That is the WAPE deliverable.

Zero-shot first (no training cost). Fine-tune (9c.1) only if zero-shot is
competitive (CV-WAPE < ~16) - the `chronos-2-native` LB precedent (14.93 fine-tuned)
says the headroom is in fine-tuning.

**Runtime:** needs a GPU (Kaggle T4 is enough; ~300 series/s zero-shot). Each fold
forecasts 96 series x 336h.


In [11]:
import torch
print(torch.__version__, torch.version.cuda,
      torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
print((torch.randn(8, device="cuda") + 1).sum().item())   # must not error


2.10.0+cu128 12.8 Tesla T4 (7, 5)
4.589629173278809


---
## §0. Configuration

In [12]:
from pathlib import Path

DATA_DIR   = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR = Path("/kaggle/working")
# Local:
# DATA_DIR   = Path("data")
# OUTPUT_DIR = Path("experiment-9")

# OOF from the pure-DL gated MLP (exp-7e) - the stacking partner. Upload on Kaggle.
MLP_OOF_PATH = Path("/kaggle/input/datasets/xitij27/exp7-oof/oof_predictions_7e_mlp_gated.csv")
# Local: Path("experiment-7/oof_predictions_7e_mlp_gated.csv")

TRAIN_PATH     = DATA_DIR / "train.csv"
VAL_INPUT_PATH = DATA_DIR / "validation_input.csv"

SEED         = 42
N_FOLDS      = 3
EVAL_HORIZON = 336

CHRONOS_MODEL   = "amazon/chronos-2"
QUANTILE_LEVELS = [0.1, 0.5, 0.9]   # 0.5 = point forecast for WAPE; 0.9 for the L2/spike view

DOMAIN_FEAT_COLS = [
    "workload_intensity", "demand_forecast", "upstream_quality_forecast",
    "promotion_intensity", "shock_risk", "maintenance_known",
    "unit_reliability_forecast", "queue_pressure_forecast",
    "network_pressure_forecast", "event_load_forecast",
    "service_irregularity_risk_forecast",
]
# Covariates passed to Chronos as KNOWN-FUTURE inputs (auto-detected from shared columns).
COVARIATE_COLS = DOMAIN_FEAT_COLS

TIME_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos", "is_weekend"]
TOP_DOM   = ["queue_pressure_forecast", "network_pressure_forecast",
             "workload_intensity", "shock_risk", "event_load_forecast"]
TOP3_ROLL = TOP_DOM[:3]
LAGS_DOM, LEADS_DOM, ROLL_WINS = [1, 24], [1, 24], [6, 24]
BASELINE_LGBM = dict(
    n_estimators=600, learning_rate=0.03, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
    reg_lambda=1.0, verbose=-1,
)

# Decision thresholds - pre-registered
ANCHOR_LGBM        = 14.78
BEATS_MLP          = 14.52
BEATS_GATED        = 14.59
BEATS_ENSEMBLE     = 14.47   # the 14.47 LGBM+MLP CV blend (7b) - the bar for the WAPE deliverable
BLEND_R_THRESHOLD  = 0.85    # Chronos is a useful stacking arm iff residual r < this
ZS_COMPETITIVE     = 16.0    # if zero-shot CV-WAPE < this, fine-tuning (9c.1) is worth it
MSE_TARGET_OOF     = 8.30    # chronos-competitive L2 (cf. 8a)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR     = {DATA_DIR}")
print(f"model        = {CHRONOS_MODEL}, quantiles {QUANTILE_LEVELS}")
print(f"covariates   = {len(COVARIATE_COLS)} domain forecasts (known future)")
print(f"decisions    : blend arm iff r < {BLEND_R_THRESHOLD}; WAPE deliverable must beat {BEATS_ENSEMBLE}")


DATA_DIR     = /kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data
model        = amazon/chronos-2, quantiles [0.1, 0.5, 0.9]
covariates   = 11 domain forecasts (known future)
decisions    : blend arm iff r < 0.85; WAPE deliverable must beat 14.47


---
## §1. Setup & data loading

In [13]:
import subprocess, sys
# Install Chronos-2 WITHOUT upgrading Kaggle's CUDA/torch stack.
# The default install clobbers cuda-core/numba-cuda runtime libs and breaks torch
# ("CUDA error: no kernel image available"). --no-deps keeps the working GPU stack.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "chronos-forecasting>=2.0", "jaxtyping"], check=False)
# Chronos-2 needs these exact ranges; --no-deps above skipped them. Pure-python
# (no cuda-core/numba-cuda) so they will NOT re-break the GPU. WITHOUT them the model
# loads with RANDOM weights -> flat forecasts (the 9c v1/v2 failure).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.41,<5", "einops>=0.7.0", "accelerate>=0.34,<2"], check=False)
# If import below raises ModuleNotFoundError for a pure-python pkg, add it with --no-deps too.
# (Kaggle already ships compatible torch / transformers / accelerate.)

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
import torch
from chronos import Chronos2Pipeline

np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
if DEVICE != "cuda":
    print("  WARNING: Chronos-2 is slow on CPU. Use a GPU runtime.")

def wape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + eps) * 100

train_df = pd.read_csv(TRAIN_PATH, parse_dates=["timestamp"])
sid_to_code = {sid: i for i, sid in enumerate(sorted(train_df["series_id"].unique()))}
code_to_sid = {v: k for k, v in sid_to_code.items()}
N_SERIES = len(sid_to_code)
print(f"train_df: {train_df.shape}, # series: {N_SERIES}")


device: cuda
train_df: (414720, 25), # series: 96


---
## §2. Feature engineering - shared tabular pipeline

The full 52-feature pipeline feeds the **LGBM anchor** (for the residual-r diagnostic, identical to 8a/9a). Chronos consumes only the imputed raw target history + the 11 domain covariates - no engineered lags/leads.

In [14]:
def impute_domain_features(df, cols):
    df = df.sort_values(["series_id", "timestamp"]).copy()
    df[cols] = df.groupby("series_id", group_keys=False)[cols].transform(lambda x: x.ffill().bfill())
    return df

def add_time_features(df, time_col="timestamp"):
    df = df.copy(); t = pd.to_datetime(df[time_col])
    df["hour_sin"]=np.sin(2*np.pi*t.dt.hour/24); df["hour_cos"]=np.cos(2*np.pi*t.dt.hour/24)
    df["dow_sin"]=np.sin(2*np.pi*t.dt.dayofweek/7); df["dow_cos"]=np.cos(2*np.pi*t.dt.dayofweek/7)
    df["doy_sin"]=np.sin(2*np.pi*t.dt.dayofyear/365); df["doy_cos"]=np.cos(2*np.pi*t.dt.dayofyear/365)
    df["is_weekend"]=(t.dt.dayofweek>=5).astype(float)
    return df

def add_temporal_features(df, lag_cols, roll_cols, lags, leads, windows):
    df = df.copy().sort_values(["series_id", "timestamp"]); new=[]
    for col in lag_cols:
        g = df.groupby("series_id", sort=False)[col]
        for L in lags:  c=f"{col}_lag{L}";  df[c]=g.shift(L);  new.append(c)
        for L in leads: c=f"{col}_lead{L}"; df[c]=g.shift(-L); new.append(c)
    for col in roll_cols:
        g = df.groupby("series_id", sort=False)[col]
        for W in windows:
            cm=f"{col}_roll{W}m"; df[cm]=g.shift(1).transform(lambda x:x.rolling(W,min_periods=1).mean()); new.append(cm)
            cs=f"{col}_roll{W}s"; df[cs]=g.shift(1).transform(lambda x:x.rolling(W,min_periods=1).std().fillna(0)); new.append(cs)
    return df, new

base_tab = add_time_features(impute_domain_features(train_df, DOMAIN_FEAT_COLS))
base_tab["sid_code"] = base_tab["series_id"].map(sid_to_code).astype("int32")
base_tab, TEMPORAL_FEATS = add_temporal_features(
    base_tab, lag_cols=TOP_DOM, roll_cols=TOP3_ROLL, lags=LAGS_DOM, leads=LEADS_DOM, windows=ROLL_WINS)
NUMERIC_FEATS  = DOMAIN_FEAT_COLS + TIME_COLS + ["series_mean"] + TEMPORAL_FEATS
FEATS_WITH_SID = NUMERIC_FEATS + ["sid_code"]
print(f"# tabular features (LGBM anchor): {len(FEATS_WITH_SID)}")
print(f"Chronos covariates: {COVARIATE_COLS}")


# tabular features (LGBM anchor): 52
Chronos covariates: ['workload_intensity', 'demand_forecast', 'upstream_quality_forecast', 'promotion_intensity', 'shock_risk', 'maintenance_known', 'unit_reliability_forecast', 'queue_pressure_forecast', 'network_pressure_forecast', 'event_load_forecast', 'service_irregularity_risk_forecast']


---
## §3. CV framework + LGBM-MAE anchor

Same rolling-origin folds and anchor as every prior experiment, so the residual-r diagnostic and the MLP-OOF merge align row-for-row.

In [15]:
def fold_offset(k): return (N_FOLDS - k) * EVAL_HORIZON

def fold_split_tabular(end_off, length=EVAL_HORIZON):
    tr_list, te_list = [], []
    for _, g in base_tab.groupby("series_id", sort=False):
        g = g.sort_values("timestamp"); cut = len(g) - end_off
        tr_list.append(g.iloc[:cut]); te_list.append(g.iloc[cut:cut+length])
    tr = pd.concat(tr_list, ignore_index=True); te = pd.concat(te_list, ignore_index=True)
    sm = tr.groupby("series_id")["target"].mean().rename("series_mean").reset_index()
    return tr.merge(sm, on="series_id"), te.merge(sm, on="series_id")

print("LightGBM-MAE anchor...")
oof_lgbm_rows, per_fold_lgbm = [], []
for k in range(N_FOLDS):
    tr, te = fold_split_tabular(fold_offset(k))
    m = lgb.LGBMRegressor(**BASELINE_LGBM, objective="regression_l1", random_state=SEED)
    m.fit(tr[FEATS_WITH_SID], tr["target"].to_numpy(float), categorical_feature=["sid_code"])
    p = m.predict(te[FEATS_WITH_SID])
    per_fold_lgbm.append(wape(te["target"], p))
    te = te.copy(); te["lead_h"] = te.groupby("series_id").cumcount() + 1
    oof_lgbm_rows.append(te.assign(pred_lgbm=p, fold=k)[
        ["series_id","timestamp","target","pred_lgbm","lead_h","fold"]])
oof_lgbm = pd.concat(oof_lgbm_rows, ignore_index=True)
LGBM_WAPE = float(np.mean(per_fold_lgbm))
assert abs(LGBM_WAPE - ANCHOR_LGBM) < 0.5, f"anchor drifted: {LGBM_WAPE:.3f}"
print(f"OK anchor reproduces: {LGBM_WAPE:.3f}")


LightGBM-MAE anchor...
OK anchor reproduces: 14.782


---
## §4. Chronos-2 zero-shot forecasting

Per fold: `context_df` = all history before the test window (target + 11 covariates); `future_df` = the 336 test rows with covariates only (no target). Chronos auto-detects the covariates from the shared columns and returns quantile forecasts; the `0.5` column is the point forecast for WAPE.

In [16]:
pipe = Chronos2Pipeline.from_pretrained(CHRONOS_MODEL, device_map=DEVICE)

CTX_COLS = ["series_id", "timestamp", "target"] + COVARIATE_COLS
FUT_COLS = ["series_id", "timestamp"] + COVARIATE_COLS

def chronos_predict(context_df, future_df=None):
    "predict_df wrapper; future_df=None -> univariate (history-only) forecast."
    kw = dict(prediction_length=EVAL_HORIZON, quantile_levels=QUANTILE_LEVELS,
              id_column="series_id", timestamp_column="timestamp", target="target")
    if future_df is not None:
        return pipe.predict_df(context_df, future_df=future_df, **kw)
    return pipe.predict_df(context_df, **kw)

def median_col(pred_df):
    if "0.5" in pred_df.columns: return "0.5"
    if "predictions" in pred_df.columns: return "predictions"
    return pred_df.columns[-1]

print("pipe ready;", len(COVARIATE_COLS), "covariates")

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

pipe ready; 11 covariates


### §4.0 Model integrity check (run this FIRST)

v1/v2 forecast flat because the model loaded with **random weights** (transformers/einops mismatch from `--no-deps`; the warning was hidden by `filterwarnings`). This cell prints dep versions, **reloads with warnings visible** (watch for *'some weights were newly initialized'* = broken), and runs a **sine-wave sanity test**: healthy model tracks the sine (pred std ~3-4); broken one stays flat (~0).

In [ ]:
import importlib.metadata as M, transformers, einops, warnings, logging
print("transformers", transformers.__version__, "| einops", einops.__version__,
      "| chronos-forecasting", M.version("chronos-forecasting"))

warnings.simplefilter("default")
logging.getLogger("transformers.modeling_utils").setLevel(logging.WARNING)
pipe = Chronos2Pipeline.from_pretrained(CHRONOS_MODEL, device_map=DEVICE)  # watch for "newly initialized"

ts  = pd.date_range("2023-01-01", periods=512, freq="h")
sig = 10 + 5*np.sin(2*np.pi*np.arange(512)/24)
demo = pd.DataFrame({"series_id": "demo", "timestamp": ts, "target": sig})
pdf = chronos_predict(demo[["series_id","timestamp","target"]], None)
mc = median_col(pdf)
print(f"\nsine-test pred std = {pdf[mc].std():.3f}   (HEALTHY ~3-4  |  BROKEN ~0)")
print("first 12 preds:", pdf[mc].round(2).head(12).tolist())
assert pdf[mc].std() > 1.0, "MODEL BROKEN (flat on a sine) -> fix transformers/einops, restart, rerun."
print("\nOK model tracks the sine - weights loaded. Proceed to 4a/4b/4c.")

### §4a. Sanity check on ONE series - does Chronos condition on history?

9c v1 produced a flat forecast (within-series std 0.008) with collapsed quantiles. Compare covariate vs univariate on a single series: a healthy forecast has **std >> 0** (tracks the daily cycle) and **non-zero quantile spread**.

In [17]:
tr0, te0 = fold_split_tabular(fold_offset(0))
one = sorted(tr0["series_id"].unique())[0]
ctx1     = tr0[tr0.series_id == one][CTX_COLS].sort_values("timestamp")
fut1     = te0[te0.series_id == one][FUT_COLS].sort_values("timestamp")
ctx1_uni = ctx1[["series_id", "timestamp", "target"]]
tgt_std  = te0[te0.series_id == one]["target"].std()

for label, cdf, fdf in [("with covariates", ctx1, fut1), ("univariate", ctx1_uni, None)]:
    pdf = chronos_predict(cdf, fdf)
    mc = median_col(pdf); pp = pdf[mc].to_numpy()
    print(f"\n[{label}] cols={list(pdf.columns)}")
    print(f"   pred std within series = {np.std(pp):.3f}   (target std = {tgt_std:.3f})")
    print(f"   pred range = [{pp.min():.2f}, {pp.max():.2f}]")
    if "0.1" in pdf.columns and "0.9" in pdf.columns:
        print(f"   mean quantile spread q90-q10 = {(pdf['0.9']-pdf['0.1']).mean():.3f}  (0 = collapsed)")
print(f"\n{one} actual target head:",
      te0[te0.series_id == one].sort_values('timestamp')['target'].head(8).round(2).tolist())


[with covariates] cols=['series_id', 'timestamp', 'target_name', 'predictions', '0.1', '0.5', '0.9']
   pred std within series = 0.008   (target std = 4.781)
   pred range = [9.06, 9.10]
   mean quantile spread q90-q10 = 0.003  (0 = collapsed)

[univariate] cols=['series_id', 'timestamp', 'target_name', 'predictions', '0.1', '0.5', '0.9']
   pred std within series = 0.007   (target std = 4.781)
   pred range = [9.06, 9.10]
   mean quantile spread q90-q10 = 0.003  (0 = collapsed)

unit_000 actual target head: [1.43, 2.13, 1.85, 2.2, 3.21, 3.09, 10.36, 10.04]


### §4b. Univariate (history-only) control across all folds

Establishes the no-covariate baseline. **If this is healthy (~17-20 WAPE) but the covariate run is ~40, the covariates are the culprit** (scale/auto-detection). If this is *also* flat/40, the problem is context/frequency, not covariates.

In [18]:
uni_fold = []
for k in range(N_FOLDS):
    tr, te = fold_split_tabular(fold_offset(k))
    ctx = tr[["series_id", "timestamp", "target"]].sort_values(["series_id", "timestamp"])
    pdf = chronos_predict(ctx, None)
    mc = median_col(pdf); pdf = pdf.rename(columns={mc: "pred_uni"})
    m = te[["series_id", "timestamp", "target"]].merge(
        pdf[["series_id", "timestamp", "pred_uni"]], on=["series_id", "timestamp"], how="left")
    m["pred_uni"] = m["pred_uni"].clip(lower=0)
    uni_fold.append(wape(m["target"], m["pred_uni"]))
    print(f"  fold {k}: univariate WAPE = {uni_fold[-1]:.3f}")
print(f"\nUnivariate CV-WAPE = {np.mean(uni_fold):.3f}   (covariate v1 was 40.97; older history-only Chronos LB ~17-20)")

  fold 0: univariate WAPE = 39.246
  fold 1: univariate WAPE = 43.888
  fold 2: univariate WAPE = 39.768

Univariate CV-WAPE = 40.967   (covariate v1 was 40.97; older history-only Chronos LB ~17-20)


### §4c. Covariate run - full CV (the main forecast)

Only trust this once §4a/§4b show Chronos is actually conditioning on history.

In [19]:
oof_chronos_rows, per_fold_chronos = [], []
for k in range(N_FOLDS):
    tr, te = fold_split_tabular(fold_offset(k))
    context_df = tr[CTX_COLS].sort_values(["series_id", "timestamp"]).reset_index(drop=True)
    future_df  = te[FUT_COLS].sort_values(["series_id", "timestamp"]).reset_index(drop=True)

    pred_df = chronos_predict(context_df, future_df)
    mc = median_col(pred_df)
    pred_df = pred_df.rename(columns={mc: "pred_chronos"})
    pred_df["pred_chronos_q90"] = pred_df["0.9"] if "0.9" in pred_df.columns else pred_df["pred_chronos"]
    pred_df["pred_chronos"] = pred_df["pred_chronos"].clip(lower=0)

    merged = te[["series_id", "timestamp", "target"]].merge(
        pred_df[["series_id", "timestamp", "pred_chronos", "pred_chronos_q90"]],
        on=["series_id", "timestamp"], how="left")
    merged["lead_h"] = merged.groupby("series_id").cumcount() + 1
    merged["fold"] = k
    per_fold_chronos.append(wape(merged["target"], merged["pred_chronos"]))
    oof_chronos_rows.append(merged)
    print(f"  fold {k}: Chronos (covariate) WAPE = {per_fold_chronos[-1]:.3f}")

oof_chronos = pd.concat(oof_chronos_rows, ignore_index=True)
CHRONOS_WAPE = float(np.mean(per_fold_chronos))
print(f"\nChronos-2 covariate CV-WAPE = {CHRONOS_WAPE:.3f}  (anchor {LGBM_WAPE:.3f}, gated-MLP {BEATS_GATED})")

  fold 0: Chronos (covariate) WAPE = 39.246
  fold 1: Chronos (covariate) WAPE = 43.894
  fold 2: Chronos (covariate) WAPE = 39.769

Chronos-2 covariate CV-WAPE = 40.970  (anchor 14.782, gated-MLP 14.59)


---
## §5. Evaluation - metrics, residual decorrelation, per-horizon

The decorrelation numbers (r vs LGBM and vs the gated MLP) are the decisive output: even if Chronos's solo WAPE trails, r < 0.85 makes it a real stacking arm.

In [20]:
def all_metrics(y, p):
    err = np.asarray(y,float) - np.asarray(p,float)
    return dict(MAE=float(np.mean(np.abs(err))), MSE=float(np.mean(err**2)),
                RMSE=float(np.sqrt(np.mean(err**2))),
                WAPE=float(100*np.sum(np.abs(err))/np.sum(np.abs(y))))

# bring in the gated-MLP OOF (the stacking partner)
have_mlp = MLP_OOF_PATH.exists()
J = oof_lgbm.merge(oof_chronos[["series_id","timestamp","fold","pred_chronos","pred_chronos_q90"]],
                   on=["series_id","timestamp","fold"], how="inner")
if have_mlp:
    mlp = pd.read_csv(MLP_OOF_PATH, parse_dates=["timestamp"])
    J = J.merge(mlp[["series_id","timestamp","fold","pred_mlp_gated"]],
                on=["series_id","timestamp","fold"], how="inner")
assert len(J) == len(oof_lgbm), "OOF merge dropped rows."

y = J["target"].to_numpy()
panel = [{"model":"LGBM-MAE", **all_metrics(y, J["pred_lgbm"])},
         {"model":"Chronos-2 zs", **all_metrics(y, J["pred_chronos"])}]
if have_mlp:
    panel.insert(1, {"model":"gated-MLP", **all_metrics(y, J["pred_mlp_gated"])})
print(pd.DataFrame(panel).to_string(index=False, float_format=lambda x:f"{x:.4f}"))

def res(col): return J["target"] - J[col]
r_lgbm = float(np.corrcoef(res("pred_lgbm"), res("pred_chronos"))[0,1])
print(f"\nr(Chronos resid, LGBM resid) = {r_lgbm:.4f}  "
      f"({'DECORRELATED - real arm' if r_lgbm < BLEND_R_THRESHOLD else 'correlated'})")
if have_mlp:
    r_mlp = float(np.corrcoef(res("pred_mlp_gated"), res("pred_chronos"))[0,1])
    print(f"r(Chronos resid, gatedMLP resid) = {r_mlp:.4f}  "
          f"({'DECORRELATED - real arm' if r_mlp < BLEND_R_THRESHOLD else 'correlated'})")

print("\nPer-horizon WAPE:")
for lo,hi in [(1,1),(2,6),(7,24),(25,72),(73,168),(169,336)]:
    sl = J[(J["lead_h"]>=lo)&(J["lead_h"]<=hi)]
    cols = ["pred_lgbm","pred_chronos"] + (["pred_mlp_gated"] if have_mlp else [])
    vals = "  ".join(f"{c.split('_')[-1]}={wape(sl['target'],sl[c]):.2f}" for c in cols)
    print(f"  {lo:>3}..{hi:<3}h (n={len(sl):>5}): {vals}")


       model    MAE     MSE   RMSE    WAPE
    LGBM-MAE 1.5546  8.8663 2.9776 14.7291
   gated-MLP 1.5349  8.7478 2.9577 14.5424
Chronos-2 zs 4.3063 31.1754 5.5835 40.8007

r(Chronos resid, LGBM resid) = 0.5376  (DECORRELATED - real arm)
r(Chronos resid, gatedMLP resid) = 0.5282  (DECORRELATED - real arm)

Per-horizon WAPE:
    1..1  h (n=  288): lgbm=24.05  chronos=114.85  gated=25.69
    2..6  h (n= 1440): lgbm=21.57  chronos=87.34  gated=21.48
    7..24 h (n= 5184): lgbm=12.51  chronos=37.96  gated=12.26
   25..72 h (n=13824): lgbm=17.18  chronos=43.08  gated=16.77
   73..168h (n=27648): lgbm=14.05  chronos=40.09  gated=13.85
  169..336h (n=48384): lgbm=14.67  chronos=40.04  gated=14.55


---
## §6. Stacking - OOF-tuned blend (the WAPE deliverable)

Grid-search blend weights on the OOF simplex, minimising per-fold mean WAPE. 2-way (MLP+Chronos) and 3-way (LGBM+MLP+Chronos). The bar is the 14.47 ensemble.

In [21]:
def cv_wape_blend(weights, cols):
    pred = sum(w * J[c].to_numpy() for w, c in zip(weights, cols))
    J["_b"] = pred; pf=[wape(g["target"], g["_b"]) for _,g in J.groupby("fold")]
    J.drop(columns="_b", inplace=True); return float(np.mean(pf))

print(f"anchor LGBM {LGBM_WAPE:.3f} | Chronos zs {CHRONOS_WAPE:.3f}", end="")
if have_mlp: print(f" | gated-MLP {all_metrics(y, J['pred_mlp_gated'])['WAPE']:.3f}")
else: print()

if have_mlp:
    # 2-way MLP + Chronos
    grid = np.round(np.arange(0,1.0001,0.025),3)
    best2 = min(((w, cv_wape_blend([w,1-w], ["pred_mlp_gated","pred_chronos"])) for w in grid), key=lambda t:t[1])
    print(f"\n2-way  w*gatedMLP + (1-w)*Chronos : best {best2[1]:.3f} at w={best2[0]:.3f}  (Δ vs 14.47 ensemble: {BEATS_ENSEMBLE-best2[1]:+.3f})")
    # 3-way LGBM + MLP + Chronos
    best3 = (None, 1e9)
    for a in grid:
        for b in grid:
            if a+b > 1: continue
            w = cv_wape_blend([a,b,1-a-b], ["pred_lgbm","pred_mlp_gated","pred_chronos"])
            if w < best3[1]: best3 = ((a,b,round(1-a-b,3)), w)
    print(f"3-way  (LGBM, gatedMLP, Chronos) : best {best3[1]:.3f} at w={best3[0]}  (Δ vs 14.47: {BEATS_ENSEMBLE-best3[1]:+.3f})")
else:
    print("\n(gated-MLP OOF not found - set MLP_OOF_PATH to run the stacking study.)")


anchor LGBM 14.782 | Chronos zs 40.970 | gated-MLP 14.542

2-way  w*gatedMLP + (1-w)*Chronos : best 14.593 at w=1.000  (Δ vs 14.47 ensemble: -0.123)
3-way  (LGBM, gatedMLP, Chronos) : best 14.514 at w=(np.float64(0.35), np.float64(0.65), np.float64(0.0))  (Δ vs 14.47: -0.044)


---
## §7. Save artifacts + decision summary

In [22]:
oof_out = oof_chronos[["series_id","timestamp","target","pred_chronos","pred_chronos_q90","lead_h","fold"]]
oof_path = OUTPUT_DIR / "oof_predictions_9c_chronos.csv"
oof_out.to_csv(oof_path, index=False)
print(f"OOF -> {oof_path}")

chronos_mse = all_metrics(y, J["pred_chronos"])["MSE"]
print("\n=== Decision summary (Frontier 2 - Chronos-2 zero-shot) ===")
print(f"  Chronos zero-shot WAPE : {CHRONOS_WAPE:.3f}")
print(f"  Chronos OOF MSE        : {chronos_mse:.3f}  (target <= {MSE_TARGET_OOF}; chronos-2-native LB ~7.72)")
print(f"  r vs LGBM              : {r_lgbm:.3f}  (decorrelated if < {BLEND_R_THRESHOLD})")

decorrelated = r_lgbm < BLEND_R_THRESHOLD
competitive  = CHRONOS_WAPE < ZS_COMPETITIVE
print("\n=== Verdict ===")
if decorrelated:
    print("DECORRELATED: Chronos is a genuine stacking arm (unlike the r~0.99 within-family).")
    print("  -> ship the best blend from §6 if it beats 14.47; this is the WAPE deliverable.")
if competitive:
    print(f"COMPETITIVE zero-shot ({CHRONOS_WAPE:.3f} < {ZS_COMPETITIVE}).")
    print("  -> 9c.1: FINE-TUNE Chronos-2 with covariates (chronos-forecasting>=2.1).")
    print("     The chronos-2-native LB precedent (14.93 fine-tuned) says the headroom is here.")
if not decorrelated and not competitive:
    print("Chronos neither decorrelated nor zero-shot-competitive on this data.")
    print("  -> document and reconsider; fine-tuning may still help but EV is lower.")


OOF -> /kaggle/working/oof_predictions_9c_chronos.csv

=== Decision summary (Frontier 2 - Chronos-2 zero-shot) ===
  Chronos zero-shot WAPE : 40.970
  Chronos OOF MSE        : 31.175  (target <= 8.3; chronos-2-native LB ~7.72)
  r vs LGBM              : 0.538  (decorrelated if < 0.85)

=== Verdict ===
DECORRELATED: Chronos is a genuine stacking arm (unlike the r~0.99 within-family).
  -> ship the best blend from §6 if it beats 14.47; this is the WAPE deliverable.
